# 03 - Feature Engineering: The Early Warning Engine

This notebook calculates the core predictive features required for the ML model, focusing on IFRS 9 quantitative triggers and behavioral distress signals.

In [4]:
import pandas as pd
import numpy as np

# Assuming df is passed from preprocessing
df = pd.read_csv('../data/raw/banking_finance_01.csv') # Placeholder raw for demo


### 1. IFRS 9 Math Triggers (PD Change & DPD Buckets)

In [5]:
# Ensure relative change matches the theoretical definition
df['calc_pd_relative_change'] = ((df['pd_12m_current_pct'] - df['pd_12m_at_origination_pct']) / df['pd_12m_at_origination_pct']) * 100

# Create categorical DPD buckets
bins = [-1, 0, 6, 13, 29, 999]
labels = ['0 DPD', '1-6 DPD', '7-13 DPD', '14-29 DPD', '30+ DPD']
df['dpd_bucket'] = pd.cut(df['days_past_due_current'], bins=bins, labels=labels)

print(df.groupby('dpd_bucket')['stage_2_migration'].mean() * 100)

dpd_bucket
0 DPD        15.871966
1-6 DPD      58.776645
7-13 DPD     71.769815
14-29 DPD    80.229638
Name: stage_2_migration, dtype: float64


### 2. Behavioral Distress Index
Combining multiple weak signals into a strong composite index.

In [6]:
df['distress_score'] = 0
df.loc[df['payment_reduction_requested'] == 'Yes', 'distress_score'] += 3
df.loc[df['forbearance_history'] == 'Yes', 'distress_score'] += 2
df.loc[df['current_account_debit_flag'] == 'Yes', 'distress_score'] += 2
df['distress_score'] += df['missed_payments_last_12m']

print(df.groupby('distress_score')['stage_2_migration'].mean() * 100)

distress_score
0       6.459518
1      30.807653
2      21.794315
3      41.152982
4      58.764892
5      56.708861
6      79.462875
7      82.299546
8      90.390390
9      94.495413
10    100.000000
11    100.000000
Name: stage_2_migration, dtype: float64
